In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

len(v1)

384

In [6]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [7]:
v1.dot(dv)

np.float32(0.32332402)

In [8]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [9]:
v2.dot(dv)

np.float32(0.019730477)

In [10]:
from ingest import load_faq_data

documents = load_faq_data()

In [11]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [12]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [13]:
import numpy as np
X = np.array(vectors)

In [15]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)


scores = X.dot(v_query)

In [16]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [17]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294094))

In [18]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [19]:
top5 = np.argsort(scores)[-5:]

In [24]:
top5 = top5[::-1]
top5



array([  7, 538, 907, 625,   2])

In [26]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.56009996
{'id': '068529125b', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I follow the course after it finishes?', 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.'}

0.6536312
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date

In [27]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [28]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [29]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [30]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)